# PoC: Visual RAG com ColPali (SOTA)

**Objetivo**: Validar a ingestão end-to-end baseada em visão para documentos clínicos complexos (tabelas, fluxogramas, ECGs) usando **ColPali** através da biblioteca `Byaldi`.

**Restrições Arquiteturais Mapeadas**:
1. **VRAM (6GB Max)**: O modelo original `vidore/colpali-v1.2` (baseado no PaliGemma de 3B parâmetros) exige ~6GB em bf16. Para rodar na RTX 2060 de forma segura, faremos quantização usando `bitsandbytes` (`load_in_4bit=True`), derrubando o uso para ~2.5GB.
2. **Espaço em Disco**: O paradigma de *Late Interaction* (ColBERT) salva múltiplos vetores por página. O custo de armazenamento é de cerca de **15 a 20 MB por página de PDF**. Indexar bibliotecas imensas requer planejamento de infraestrutura (NVMe ou buckets de baixo custo).
3. **Tamanho do Batch**: Processaremos a indexação de forma unitária (batch pequeno) para não estourar a VRAM durante a geração de vetores.

In [1]:
print("start")

start


In [2]:
print(f"ETA esperado: 7min")

import os
from dotenv import load_dotenv
os.environ['HF_HOME'] = '/mnt/gamer_d/huggingface_cache'
os.environ['HF_HUB_CACHE'] = '/mnt/gamer_d/huggingface_cache'
load_dotenv("../env/creds.env")

import torch
from IPython.display import Image, display
from byaldi import RAGMultiModalModel

# Carrega as credenciais do arquivo centralizado
# O caminho é relativo à pasta 'playground/'
load_dotenv("../env/creds.env")

print(f"CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")

ETA esperado: 7min
CUDA Disponível: True
Dispositivo: NVIDIA GeForce RTX 2060


### 1. Inicialização do Modelo (Gestão de Memória)
Injetando os parâmetros do HuggingFace para forçar a quantização 4-bit na GPU.

In [ ]:
# run once in terminal
# HF_HUB_ENABLE_HF_TRANSFER=1 uv run hf download vidore/colpaligemma-3b-pt-448-base --cache-dir /mnt/gamer_d/huggingface_cache


In [ ]:
# =============================================================================
# PASTE THIS into the model-loading cell of colpali_poc.ipynb
# (replace the current cell that starts with "# Configurações para proteger a VRAM")
# =============================================================================

import time
from pathlib import Path
from colpali_engine.models import ColPali, ColPaliProcessor
from transformers import BitsAndBytesConfig

# Path to the pre-saved merged+quantized model
MERGED_MODEL_PATH = Path(os.path.abspath("../models/colpali-v1.2-merged-4bit"))
ADAPTER_REPO = "vidore/colpali-v1.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

start = time.time()

if MERGED_MODEL_PATH.exists() and (MERGED_MODEL_PATH / "config.json").exists():
    # === FAST PATH: Load pre-saved merged model from disk ===
    print(f"⚡ Loading pre-saved quantized model from {MERGED_MODEL_PATH}...")
    model = ColPali.from_pretrained(
        str(MERGED_MODEL_PATH),
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        local_files_only=True,
    ).eval()
    processor = ColPaliProcessor.from_pretrained(
        str(MERGED_MODEL_PATH),
        local_files_only=True,
    )
else:
    # === FIRST RUN: Load adapter, merge, save locally ===
    print(f"🔧 First run: loading adapter from '{ADAPTER_REPO}', merging & saving...")
    model = ColPali.from_pretrained(
        ADAPTER_REPO,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    ).eval()
    processor = ColPaliProcessor.from_pretrained(ADAPTER_REPO)

    # Save merged model + processor to disk for future instant loads
    print(f"💾 Saving merged model to {MERGED_MODEL_PATH}...")
    MERGED_MODEL_PATH.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(MERGED_MODEL_PATH))
    processor.save_pretrained(str(MERGED_MODEL_PATH))
    print(f"✅ Saved! Next load will be instant.")

elapsed = time.time() - start
print(f"\n✅ Model loaded in {elapsed:.1f}s")

# Wrap into Byaldi's RAG interface for indexing/search
from byaldi import RAGMultiModalModel
from byaldi.colpali import ColPaliModel

RAG = RAGMultiModalModel.__new__(RAGMultiModalModel)
inner = ColPaliModel.__new__(ColPaliModel)
inner.model = model
inner.processor = processor
inner.device = "cuda" if torch.cuda.is_available() else "cpu"
inner.verbose = 1
inner.collection = {}
inner.indexed_embeddings = []
inner.embed_id_to_doc_id = {}
inner.doc_id_to_metadata = {}
inner.doc_ids_to_file_names = {}
inner.doc_ids = set()
inner.index_name = None
inner.index_root = ".byaldi"
inner.pretrained_model_name_or_path = str(MERGED_MODEL_PATH)
inner.model_name = str(MERGED_MODEL_PATH)
inner.highest_doc_id = -1
inner.full_document_collection = False
inner.load_from_index = False
inner.kwargs = {}
inner.n_gpu = torch.cuda.device_count()
RAG.model = inner

print("🚀 RAG interface ready for indexing and search!")


Carregando ColPali (PaliGemma 3B) em 4-bits... Isso pode levar um tempo no primeiro download.
Verbosity is set to True (active). Pass verbose=0 to make quieter.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/605 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 5.60 GiB of which 19.06 MiB is free. Including non-PyTorch memory, this process has 5.34 GiB memory in use. Of the allocated memory 5.24 GiB is allocated by PyTorch, and 18.71 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### 2. Mock & Ingestão Visual
Aqui, ao invés de extrair texto, o modelo vai 'tirar uma foto' da página e extrair as feições semânticas da imagem diretamente.

In [ ]:
# Setup de teste: Criar um PDF simulado caso não tenha nenhum
from fpdf import FPDF

pdf_teste = "protocolo_visual.pdf"
if not os.path.exists(pdf_teste):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.cell(200, 10, txt="Protocolo de Dor Toracica Institucional - FLUXOGRAMA", ln=1, align='C')
    pdf.cell(200, 10, txt="Se Supra de ST no ECG -> Encaminhar para Cateterismo imediato.", ln=1, align='L')
    pdf.cell(200, 10, txt="Dose de Ataque: AAS 300mg + Ticagrelor 180mg", ln=1, align='L')
    pdf.output(pdf_teste)
    print(f"PDF mock '{pdf_teste}' gerado.")

# Ingestão (Indexação)
INDEX_NAME = "poc_dor_toracica"

print(f"Indexando o documento visualmente... (Espaço estimado: ~15MB/pág)")
RAG.index(
    input_path=pdf_teste,
    index_name=INDEX_NAME,
    store_collection_with_index=True, # Salva metadados da imagem no índice para visualização
    overwrite=True
)
print("Indexação concluída! Os tensores foram salvos no diretório .byaldi/")

### 3. Recuperação Visual (Late Interaction)
Nós fazemos uma pergunta em texto. O ColBERT vai buscar quais "patches" da imagem melhor respondem à pergunta.

In [ ]:
pergunta = "Qual a dose de ataque para IAM com Supra?"
print(f"Pesquisando: '{pergunta}'")

# Busca (k=1 pois temos só 1 página mockada)
resultados = RAG.search(pergunta, k=1)

print("\n=== Resultado ===")
for res in resultados:
    print(f"Documento: {res.doc_id}")
    print(f"Score Semântico Visual (ColBERT): {res.score:.2f}")
    print(f"Base64 Image Reference presente: {'base64' in res.metadata}")
    
    # Para debugar e mostrar ao médico de onde veio a info
    if 'base64' in res.metadata:
        from IPython.display import display, HTML
        display(HTML(f'<img src="data:image/jpeg;base64,{res.metadata["base64"]}" width="400"/>'))
